# 🎹 Activity 4 — The Piano Has Left the Building

> **Incident report, 15:47, Skyline Travel.** Edna Ivory checked in her grand piano in Vienna. She would like to know where it is. Skye looks up her booking with a **tool**. Things go downhill from there, slowly, and then all at once.

Skye decides to call the tool `get_booking`; the tool's answer goes back to Skye, who then answers Edna. The tool's settings, and Skye's instructions, are the prompt **`skye-piano`** in Langfuse.

**Rosetta stone for monolith people**
- tool call = an RPC the model decides to make, with arguments the model fills in
- tool schema = the RPC's interface definition
- `piano-mover-check` = the slow partner system behind a cache

## Purpose
Debug tool calls from the trace: wrong arguments, wrong results, and too many calls.

## What to expect
Each run prints one line and a 📎 link. The tool's input, output, errors and timing are in Langfuse.

## Bad practice
Making a tool "forgiving" so it never fails, and never looking at what it returned.

## Good practice
Read each tool step's input, output and level; mark wrong results; watch the timeline for repeated calls.

## Why the good practice works
A loud error is cheap. A quiet wrong answer, or a quiet triple call, only shows up in the trace.

## Deliverable
All missions ✅ in the final report, and the slow/fast pair of traces for the demo.

In [ ]:
import time
from experiment_utils import check, report, run_piano


def cache_key(ref):
    return f"{ref}:{time.time()}"  # how the piano movers' answers are cached (more on this in 4.3)


## 🕵️ Mission 4.1 — The wrong label

Edna asked politely. Skye apologised and asked her to try again later. Edna is 87 and does not have "later" to spare.

*Monolith reflex: the client sends `booking_ref`, the server reads `booking_id`. A 500 with a `KeyError` in the logs.*

**In Langfuse**
1. Run the cell and open the 📎 link. One step in the tree is red: **`get_booking`**.
2. Read its **Input** (the arguments Skye sent) and its error message.
3. Leave a **comment** on that step.
4. In **Prompts** → **`skye-piano`**, the **Config** decides what the tool's parameter is called. Make a new `production` version so Skye sends what the tool reads, then rerun.

In [ ]:
run_piano(cache_key)

<details><summary>💡 Stuck?</summary>

- Compare the argument name in the step's Input with the name in the error.
- `tool_param` must match what the tool reads.

</details>

In [ ]:
check("4.1")

## 🕵️ Mission 4.2 — The wrong trip

The tool no longer fails. It is very forgiving: when it cannot find a booking, it returns the first one it has. Edna has just been told her piano is a bag of fish. Somewhere, a fisherman wonders why his fish weigh 400 kg.

*Monolith reflex: a `catch` block that returns a default instead of failing. Nothing is red. Everything is wrong.*

**In Langfuse**
1. Run the cell and open the trace. Open **`get_booking`**: compare its **Input** (Edna's reference) with its **Output** (the booking it returned).
2. Mark the step as wrong with a **score**: on the step, use **Annotate**, pick the score **`correct`**, set it to **false**. Scores are how teams label good and bad results, and the basis of evaluations.
3. In **Prompts** → **`skye-piano`**, make a new `production` version whose config stops the lookup from guessing.
4. Rerun and check that the tool returns Edna's own booking.

In [ ]:
run_piano(cache_key)

<details><summary>💡 Stuck?</summary>

- Input says SKY-2087. Look at `booking_ref` in Output.
- `strict_lookup` is the switch. A loud error beats a confident wrong answer.

</details>

In [ ]:
check("4.2")

## 🕵️ Mission 4.3 — The slow lookup

Edna's piano is found. Every lookup now confirms with the piano movers' system, which is slow, so answers are cached. And yet every run takes ages.

*Monolith reflex: a cache with a 0% hit rate, and a retry loop hiding it.*

**In Langfuse**
1. Run the cell and open the trace. Switch to the **Timeline** view: where does the time go?
2. Count the **`piano-mover-check`** steps and read their **Output** (`cache: hit` or `miss`) and **Input**.
3. This fix is in code: `cache_key` in the first code cell of this notebook. Fix it, run that cell again, then rerun the cell below and compare the timelines.

In [ ]:
run_piano(cache_key)

<details><summary>💡 Stuck?</summary>

- A cache key that contains the current time never matches anything.
- The cache is keyed by booking reference.

</details>

In [ ]:
check("4.3")

## 🏁 Final report

All four activities, straight from Langfuse. If every mission is ✅, Skye is back online.

In [ ]:
report()